First of all install a dquant and MetaTrader5

In [ ]:
pip install dquant MetaTrader5

Code for basic example

In [ ]:
import pandas as pd
import MetaTrader5 as mt5
import datetime as dt
from dquant.models import VolClustXGB


# 1. Load data
symbol = "EURUSD"          # symbol to watch
timeframe = mt5.TIMEFRAME_H1   # M1, M5, M15, H1, D1, etc.
days_back = 1000             # how many days of history to load

# Connect to MT5
if not mt5.initialize():
    print("Failed to connect to MetaTrader5")
    quit()

# Check that symbol is available
if not mt5.symbol_select(symbol, True):
    print(f"Symbol {symbol} not found or not enabled")
    mt5.shutdown()
    quit()

# Calculate dates
to_date = dt.datetime.now() + dt.timedelta(hours=3)
from_date = to_date - dt.timedelta(days=days_back)

# Load bars
rates = mt5.copy_rates_range(symbol, timeframe, from_date, to_date)

mt5.shutdown()  # terminal no longer needed

if rates is None or len(rates) == 0:
    print("No data!")
    quit()

# Convert to DataFrame
df = pd.DataFrame(rates)
df['time'] = pd.to_datetime(df['time'], unit='s')

df.rename(columns={
    'tick_volume': 'volume'
}, inplace=True)


# 2. Create model
model = VolClustXGB({}, early_stopping=True)

# 3. Train model
features = [
    'TR',
    'returns',
    'abs_returns',
    'gap',
    'body',
    'shadow',
    'close_position',
    'roll_atr_14'
]
model.fit(df, feature_list=features, input_bars=70, horizon=20, trees_count=200, show_results=True)

# 4. Make forecast
rez = model.forecast(df.iloc[-70:].copy(), show=True)
